## **Introduction**

In this lab, let's play a two city game! This lab is a (very) simplified version of Sid Meier's Civilization (You can get it from Steam). Don't worry if you have not played the game before, as we will provide detailed description of the setup, which can be quite different from the original game as this would be a short lab but the original game can be quite complicated.

There are two cities on the map, each seeking to capture the other to increase its power. The only way to capture a city is by sending warriors. A warrior must be moved to the opponent's location to successfully capture that city.

The map is 7 units long, with City A located at the left (index 0) and City B at the right (index 6). Warriors can only be built within the cities, starting at their respective locations.

There are four types of actions allowed: building a warrior, building three warriors at once, building five warriors at once, or developing technology (refer to ACTION_LIST). Each action requires a different amount of energy (see ENERGY_COST). At the beginning of each round, each city receives 2 production energy by default. Developing technology can increase the energy for that specific turn by 1.

Please note that a game termination check will be conducted at the start of each round, before any actions are taken.

## **Lab work**

In [9]:
MAP_LENGTH = 7
ACTION_LIST = ['TECHNOLOGY','WARRIOR','THREE_WARRIORS','FIVE_WARRIORS']
ENERGY_COST = {'TECHNOLOGY':0,'WARRIOR':2, 'THREE_WARRIORS':5,'FIVE_WARRIORS':8}
BASE_INCOME_PER_TURN = 2
WEIGHT = [1000, 3, 2, 0, 0, 0, 0,0]
ECON_COEFF   = 6
WARRIOR_COEFF = 10

In [10]:
def new_initial_state():
    '''
    initial state
    '''
    warriors = [0]*MAP_LENGTH
    prod     = [2, 2]
    cur      = 0
    return [warriors, prod, cur]

def clone_state(state):
    '''
    make a clone of the state
    '''
    return [state[0][:], state[1][:], state[2]]

def city_index(player):
    '''
    get the location of current player's city
    '''
    if player == 0:
        return 0
    else:
        return MAP_LENGTH - 1

def opponent(player):
    '''
    get the opponent of current player
    '''
    if player == 0:
        return 1
    else:
        return 0

In [11]:
def legal_actions(state):
    '''
    get all legal actions (actions that you have enough energy to afford) for one
    player. The information about the player is stored in cur, within "state"
    '''
    warriors, prod, cur = state
    acts = []
    acts.append('TECHNOLOGY')
    if prod[cur] >= ENERGY_COST['WARRIOR']:
        acts.append('WARRIOR')
    if prod[cur] >= ENERGY_COST['THREE_WARRIORS']:
        acts.append('THREE_WARRIORS')
    if prod[cur] >= ENERGY_COST['FIVE_WARRIORS']:
        acts.append('FIVE_WARRIORS')
    return acts
def apply_action_in_place(state, action):
    '''
    apply an action and modify the states accordingly. Notice that warriors will
    not be moved here and this will be handled later.
    '''
    warriors, prod, cur = state
    prod[cur] = prod[cur] - ENERGY_COST[action]
    if action == 'WARRIOR':
        if cur == 0:
            warriors[0] = warriors[0] + 1
        else:
            warriors[MAP_LENGTH-1] = warriors[MAP_LENGTH-1] - 1
    elif action == 'THREE_WARRIORS':
        if cur == 0:
            warriors[0] = warriors[0] + 3
        else:
            warriors[MAP_LENGTH-1] = warriors[MAP_LENGTH-1] - 3
    elif action == 'FIVE_WARRIORS':
        if cur == 0:
            warriors[0] = warriors[0] + 5
        else:
            warriors[MAP_LENGTH-1] = warriors[MAP_LENGTH-1] - 5
    return state

In [ ]:
def move_warriors_in_place(state):
    '''
    Task 1: After each round, move the warriors from each city one step forward.
    Newly built warriors can also be moved.
    If a warrior from each side lands on the same tile, both warriors will eliminate each other.

    Important Notes:

    Do not move the same warrior multiple times in a single round.
    For example, if the initial state is [1, 1, 0, 0, 0, 0, 0],
    the resulting state should be [0, 1, 1, 0, 0, 0, 0], not [0, 0, 2, 0, 0, 0, 0].
    You cannot assume any specific distribution of warriors.
    Even if a state seems impossible in the current game version
    (e.g., [1, 1, 0, 0, 0, 0, 0]),
    it should still be handled correctly in your implementation.
    '''
    # Your code goes here.

    # Your code ends here.
    return state
if __name__=="__main__":
    '''
    You are expected to see:
    state[0]=[0,1,1,0,0,0,0]
    '''
    state=[[1,1,0,0,0,0,0],[0,0],0]
    state = move_warriors_in_place(state)
    print(state[0])

In [13]:
def warrior_victory(state):
    '''
    A city wins because of its warriors
    '''
    warriors, prod, cur = state
    if warriors[MAP_LENGTH-1] > 0:
        return 0
    if warriors[0] < 0:
        return 1
    return -1
def is_terminal(state):
    '''
    check whether a state is terminal state. This will happen before any action
    can be taken.
    '''
    c = warrior_victory(state)
    if c != -1:
        return True, c
    return False, -1

In [ ]:
def end_of_turn_updates_in_place(state,tech):
    '''
    handles the end of turn effects.
    Production energy will be added and the game will be passed to the opponent.
    '''
    warriors, prod, cur = state
    prod[cur] = prod[cur] + BASE_INCOME_PER_TURN
    if tech==True:
      prod[cur]=prod[cur]+1
    state[2] = opponent(cur)
    return state
def nearest_distance_to_enemy_city(warriors, player):
    '''
    calculates nearest distance of one's warrior to opponent's city. This can
    capture how likely one player would win because of its warriors.
    '''
    if player == 0:
        best = 7
        i = 0
        while i < MAP_LENGTH:
            if warriors[i] > 0:
                d = (MAP_LENGTH-1) - i
                if d < best:
                    best = d
            i = i + 1
        return best
    else:
        best = 7
        i = 0
        while i < MAP_LENGTH:
            if warriors[i] < 0:
                d = i - 0
                if d < best:
                    best = d
            i = i + 1
        return best
def evaluate(state):
    '''
    Task 2:
    In the context of mini-max and alpha-beta pruning, reaching a terminal state is ideal for easy evaluation.
    However, if it takes too long to reach a terminal state,
    we may want to stop after a few steps and evaluate a non-terminal state instead.
    In such cases, we need to define an evaluation function since there is no clear winner.

    Assume distanceA and distanceB are distance between A's nearest warriors to
    B and B's nearest warriors to A, productionA and productionB are production
    energy of A and B, warriorA and warriorB are number of warriors on the map,
    belonging to A and B, the evaluation function would be
    ECON_COEFF*(productionA-productionB)+WEIGHT[distanceA]-WEIGHT[distanceB]+
    WARRIOR_COEFF*(warriorA-warriorB).

    Notice that when one player has no warrior on the map, the distance
    between its player and the opponent is considered to be 7.
    A would maximize the evaluation function and B would minimize it.
    '''
    # Your code goes here.

    # Your code ends here.
if __name__=="__main__":
    '''
    You are expected to see:
    value=39
    '''
    state=[[1,1,0,0,0,1,0],[1,0],0]
    value = evaluate(state)
    print(value)

In [15]:
def play_one_round(state, action):
    '''
    Play the game for one round.
    '''
    st = clone_state(state)
    apply_action_in_place(st, action)
    move_warriors_in_place(st)
    end_of_turn_updates_in_place(st,(action=="TECHNOLOGY"))
    return st

In [ ]:
def minimax(state, depth):
    '''
    Task 3: Play the game using mini-max, without alpha-beta pruning.
    Write a recursive function to evaluate all possible actions and their score,
    and choose the best action. This function returns best evaluation score and
    best action for the current state. Notice that if the game terminates
    (i.e. there is a winner or depth=0) the returned best action shall be "NULL"
    and when there is a winner the returned best score should be +/- 1000, depending
    on who the winner is (A wins: +1000; B wins: -1000).

    NOTE: The depth controls how far you will go when evaluating different states.
    Decrease the depth by 1 with each recursive call.
    '''
    minimax.calls=minimax.calls+1 # DO NOT modify this line
    # Your code goes here

    # Your code ends here
    
if __name__=="__main__":
    '''
    You are expected to see:
    best_val=18
    best_action="TECHNOLOGY"
    minimax.calls=17
    '''
    state=[[0,0,0,0,0,0,0],[2,2],0]
    minimax.calls=0
    best_val, best_action = minimax(state,3)
    print(best_val)
    print(best_action)
    print(minimax.calls)

In [ ]:
def alphabeta(state, depth, alpha, beta):
    '''
    Task 4: Play the game using mini-max, with alpha-beta pruning.
    Write a recursive function to evaluate all possible actions and their score,
    and choose the best action. This function returns best evaluation score and
    best action for the current state. Notice that if the game terminates
    (i.e. there is a winner or depth=0) the returned best action shall be "NULL"
    and when there is a winner the returned best score should be +/- 1000, depending
    on who the winner is (A wins: +1000; B wins: -1000).

    NOTE: The depth controls how far you will go when evaluating different states.
    Decrease the depth by 1 with each recursive call.
    '''
    alphabeta.calls = alphabeta.calls+1 # DO NOT modify this line
    # Your code goes here

    # Your code ends here.
if __name__=="__main__":
    '''
    You are expected to see:
    best_val=18
    best_action="TECHNOLOGY"
    alphabeta.calls=12
    '''
    state=[[0,0,0,0,0,0,0],[2,2],0]
    alphabeta.calls=0
    best_val, best_action = alphabeta(state,3,-999999, 999999)
    print(best_val)
    print(best_action)
    print(alphabeta.calls)

In [ ]:
import sys
'''
Now it is time to play with your AI. Note that if the minimax is implemented
correctly, it is expected that you can never win the AI (either tie or lose).
'''
def pretty_state(state):
    warriors,prod,cur = state
    print("Warriors:", warriors)
    print("Prod    :", prod)
    print("Player  :", "A" if cur==0 else "B")
def prompt_human_action(state, player_label):
    legal = legal_actions(state)
    print(f"\nYour turn, {player_label}. Legal actions: {', '.join(legal)}")
    print("Type an action.")
    raw = input("> ").strip().upper()
    return raw
def choose_control():
    while True:
        side = input("Who do you want to control? [A/B] ").strip().upper()
        if side in {"A", "B"}:
            break
        print("Please enter A, B.")
    def is_human(turn_idx):
        return (turn_idx == 0 and side == "A") or (turn_idx == 1 and side == "B")
    depth = 3
    return is_human, depth
def visualize_state(state):
    """
    Visualize game state with map, warrior positions, and city status
    """
    warriors, prod, cur = state
    
    print("\n" + "="*50)
    print("CURRENT MAP STATE:")
    print("="*50)
    
    # Display map positions
    positions = "  0    1    2    3    4    5    6  "
    print(positions)
    
    # Display warrior distribution
    warrior_display = ""
    for i in range(MAP_LENGTH):
        if warriors[i] > 0:
            warrior_display += f" A{warriors[i]}  "
        elif warriors[i] < 0:
            warrior_display += f" B{abs(warriors[i])} "
        else:
            warrior_display += "  .  "
    print(warrior_display)
    
    # Display city markers
    cities = " CityA       Neutral      Zone        CityB "
    print(cities)
    
    # Display production energy
    print(f"\nCity A Production Energy: {prod[0]} | City B Production Energy: {prod[1]}")
    print(f"Current Turn: {'City A' if cur == 0 else 'City B'}")
    print("="*50)

def get_valid_human_action(state, player_label):
    """
    Get and validate human player input with illegal action handling
    """
    legal = legal_actions(state)
    
    while True:
        print(f"\nYour turn, {player_label}. Legal actions: {', '.join(legal)}")
        raw_input = input("Enter your action > ").strip().upper()
        
        if raw_input in legal:
            return raw_input
        else:
            print(f"INVALID ACTION: '{raw_input}' is not allowed. Please choose from: {', '.join(legal)}")

def display_final_result(winner):
    """
    Display final winning message with visual emphasis
    """
    print("\n" + "-" * 60)
    if winner == 0:
        print("VICTORY FOR CITY A!")
        print("All warriors have been eliminated!")
    else:
        print("VICTORY FOR CITY B!") 
        print("All warriors have been eliminated!")
    print("-" * 60)

def run_interactive():
    s = new_initial_state()
    is_human, ai_depth = choose_control()
    
    print("\nGame Starting!")
    visualize_state(s)
    
    for _ in range(20):
        cur_player = s[2]
        label = "A" if cur_player == 0 else "B"
        
        # Check for terminal state
        won, winner = is_terminal(s)
        if won:
            display_final_result(winner)
            break
            
        # Get action based on player type
        if is_human(cur_player):
            act = get_valid_human_action(s, label)
        else:
            val, act = alphabeta(s, ai_depth, -999999, 999999)
            print(f"AI ({label}) chooses {act} (evaluation score: {val})")
        
        # Apply action and update state
        s = play_one_round(s, act)
        visualize_state(s)
        
        # Check for terminal state after move
        won, winner = is_terminal(s)
        if won:
            display_final_result(winner)
            break
    else:
        print("\n" + "="*50)
        print("GAME ENDED: Maximum round limit reached!")
        print("This is a draw game.")
        print("="*50)
if __name__ == "__main__":
    run_interactive()